# RF-DETR Location Tag — End-to-End Training Notebook

This notebook builds a single-class RF-DETR detector for `location_tag` from JSON annotations whose `image_id` values are Azure Blob URLs.

Pipeline:
1. Read annotation JSON files
2. Normalize annotations
3. Keep a single `location_tag` class
4. Download unique Azure images in parallel
5. Create train/valid/test splits
6. Build COCO annotations
7. Create the RF-DETR dataset structure using an `images` subfolder
8. Train RF-DETR Base with `num_classes=1`, `resolution=672`, `device="cuda"`
9. Find the best checkpoint
10. Run test inference and visualize detections

## 1. Imports

In [ ]:
import os
import json
import random
import shutil
from pathlib import Path
from urllib.parse import urlparse, unquote
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

## 2. Configuration

In [ ]:
# Input / output paths
INPUT_JSON_DIR = "./coco_files"
DATASET_DIR = "./rfdetr_dataset"
OUTPUT_DIR = "./rfdetr_output"
PRETRAINED_WEIGHTS = "/home/jupyter/rf-detr-base-coco.pth"

# Azure
AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"

# Annotation fields
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"
AREA_FIELD = "area"
BBOX_FORMAT = "xywh"

# Single-class detector
TARGET_CLASS = "location_tag"
NUM_CLASSES = 1

# RF-DETR
RESOLUTION = 672
DEVICE = "cuda"
EPOCHS = 50
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 1

# Dataset split
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10
RANDOM_SEED = 42

# Download
DOWNLOAD_WORKERS = 16
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Inference
CONFIDENCE = 0.40
INFERENCE_OUTPUT_DIR = "./inference_outputs"

## 3. Install/import dependencies

In [ ]:
# Run this only if the environment does not already contain these packages.
# %pip install -U rfdetr supervision azure-storage-blob python-dotenv tqdm pillow

import torch
import supervision as sv
from rfdetr import RFDETRBase

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Azure connection

In [ ]:
load_dotenv()

connection_string = os.getenv(AZURE_CONNECTION_STRING_ENV)
if not connection_string:
    raise RuntimeError(
        f"Environment variable {AZURE_CONNECTION_STRING_ENV!r} was not found. "
        "Add it to your .env file."
    )

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
print("Azure Blob Storage client created.")

## 5. Read annotation JSON files

In [ ]:
json_paths = sorted(Path(INPUT_JSON_DIR).glob("*.json"))

if not json_paths:
    raise FileNotFoundError(f"No JSON files found in {INPUT_JSON_DIR}")

raw_records = []

for json_path in tqdm(json_paths, desc="Reading annotation JSON files"):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        raw_records.extend(data)
    elif isinstance(data, dict):
        if "annotations" in data:
            raw_records.extend(data["annotations"])
        else:
            raise ValueError(f"Unsupported JSON structure: {json_path}")
    else:
        raise ValueError(f"Unsupported JSON type: {json_path}")

print(f"JSON files: {len(json_paths)}")
print(f"Raw annotation records: {len(raw_records)}")

## 6. Normalize annotations and keep one class

In [ ]:
def normalize_category(value):
    if isinstance(value, list):
        if len(value) != 1:
            raise ValueError(f"One bbox must map to exactly one class: {value}")
        return str(value[0])
    return str(value)

normalized_annotations = []
skipped_records = 0

for record in tqdm(raw_records, desc="Processing annotations"):
    try:
        image_url = record[IMAGE_FIELD]
        category = normalize_category(record[CATEGORY_FIELD])
        bbox = record[BBOX_FIELD]

        if not image_url or not bbox or len(bbox) != 4:
            skipped_records += 1
            continue

        if category != TARGET_CLASS:
            skipped_records += 1
            continue

        x, y, w, h = [float(v) for v in bbox]

        if w <= 0 or h <= 0:
            skipped_records += 1
            continue

        # Recompute area instead of trusting the source area.
        area = w * h

        normalized_annotations.append({
            "image_id": str(image_url),
            "category": TARGET_CLASS,
            "bbox": [x, y, w, h],
            "area": area
        })

    except Exception as e:
        skipped_records += 1
        print(f"Skipping annotation: {e}")

print(f"Usable annotations: {len(normalized_annotations)}")
print(f"Skipped annotations: {skipped_records}")
print(f"Target class: {TARGET_CLASS}")

## 7. Group annotations by image URL

In [ ]:
annotations_by_image = defaultdict(list)

for annotation in normalized_annotations:
    annotations_by_image[annotation["image_id"]].append(annotation)

image_urls = sorted(annotations_by_image.keys())

print(f"Unique images: {len(image_urls)}")
print(f"Images containing {TARGET_CLASS}: {len(image_urls)}")

## 8. Azure Blob URL parser

In [ ]:
def parse_blob_url(blob_url):
    parsed = urlparse(blob_url)
    path_parts = [unquote(part) for part in parsed.path.lstrip("/").split("/") if part]

    if len(path_parts) < 2:
        raise ValueError(f"Cannot parse Azure Blob URL: {blob_url}")

    container_name = path_parts[0]
    blob_name = "/".join(path_parts[1:])

    return container_name, blob_name


def local_filename_from_url(blob_url):
    _, blob_name = parse_blob_url(blob_url)
    filename = Path(blob_name).name

    if not filename:
        raise ValueError(f"Could not determine filename from URL: {blob_url}")

    suffix = Path(filename).suffix.lower()
    if suffix not in IMAGE_EXTENSIONS:
        filename = filename + ".jpg"

    return filename

## 9. Download one image from Azure

In [ ]:
def download_one_image(image_url, destination_dir):
    try:
        container_name, blob_name = parse_blob_url(image_url)
        destination_dir = Path(destination_dir)
        destination_dir.mkdir(parents=True, exist_ok=True)

        filename = local_filename_from_url(image_url)
        destination_path = destination_dir / filename

        # Avoid overwriting if different blobs happen to have the same basename.
        if destination_path.exists():
            stem = destination_path.stem
            suffix = destination_path.suffix
            destination_path = destination_dir / f"{stem}_{abs(hash(image_url))}{suffix}"

        container_client = blob_service_client.get_container_client(container_name)
        blob_client = container_client.get_blob_client(blob_name)

        with open(destination_path, "wb") as f:
            f.write(blob_client.download_blob().readall())

        return image_url, destination_path, None

    except Exception as e:
        return image_url, None, e

## 10. Download all unique images in parallel

In [ ]:
DOWNLOAD_CACHE_DIR = Path(DATASET_DIR) / "_download_cache"
DOWNLOAD_CACHE_DIR.mkdir(parents=True, exist_ok=True)

download_results = {}
download_errors = []

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = {
        executor.submit(download_one_image, url, DOWNLOAD_CACHE_DIR): url
        for url in image_urls
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Downloading Azure images"
    ):
        image_url, local_path, error = future.result()
        download_results[image_url] = local_path

        if error is not None:
            download_errors.append((image_url, error))
            print(f"Download error: {error}")

print(f"Downloaded/available images: {sum(p is not None for p in download_results.values())}")
print(f"Download errors: {len(download_errors)}")

## 11. Remove images that could not be downloaded

In [ ]:
usable_image_urls = [
    url for url in image_urls
    if download_results.get(url) is not None
    and Path(download_results[url]).exists()
]

annotations_by_image = {
    url: annotations_by_image[url]
    for url in usable_image_urls
}

print(f"Usable images after download: {len(usable_image_urls)}")

## 12. Deterministic train / valid / test split

In [ ]:
if abs(TRAIN_RATIO + VALID_RATIO + TEST_RATIO - 1.0) > 1e-9:
    raise ValueError("TRAIN_RATIO + VALID_RATIO + TEST_RATIO must equal 1.0")

random.seed(RANDOM_SEED)

shuffled_urls = usable_image_urls.copy()
random.shuffle(shuffled_urls)

n_images = len(shuffled_urls)
train_end = int(n_images * TRAIN_RATIO)
valid_end = train_end + int(n_images * VALID_RATIO)

split_urls = {
    "train": shuffled_urls[:train_end],
    "valid": shuffled_urls[train_end:valid_end],
    "test": shuffled_urls[valid_end:]
}

for split_name, urls in split_urls.items():
    print(f"{split_name}: {len(urls)} images")

## 13. Create dataset directories

In [ ]:
dataset_root = Path(DATASET_DIR)

for split in ["train", "valid", "test"]:
    split_dir = dataset_root / split
    images_dir = split_dir / "images"

    if split_dir.exists():
        shutil.rmtree(split_dir)

    images_dir.mkdir(parents=True, exist_ok=True)

print("Dataset directories created.")

## 14. Copy downloaded images into each split/images directory

In [ ]:
split_local_paths = {}

for split_name, urls in split_urls.items():
    split_local_paths[split_name] = {}

    for image_url in tqdm(urls, desc=f"Preparing {split_name} images"):
        source_path = Path(download_results[image_url])
        destination_dir = dataset_root / split_name / "images"

        filename = source_path.name
        destination_path = destination_dir / filename

        if destination_path.exists():
            stem = destination_path.stem
            suffix = destination_path.suffix
            destination_path = destination_dir / f"{stem}_{abs(hash(image_url))}{suffix}"

        shutil.copy2(source_path, destination_path)
        split_local_paths[split_name][image_url] = destination_path

print("Images copied into train/valid/test images folders.")

## 15. COCO helper

In [ ]:
def build_coco_for_split(split_name, image_urls):
    images = []
    annotations = []

    image_id_map = {}

    for index, image_url in enumerate(image_urls, start=1):
        image_path = split_local_paths[split_name][image_url]

        with Image.open(image_path) as image:
            width, height = image.size

        image_id_map[image_url] = index

        images.append({
            "id": index,
            "file_name": f"images/{image_path.name}",
            "width": width,
            "height": height
        })

    annotation_id = 1

    for image_url in image_urls:
        image_id = image_id_map[image_url]

        for item in annotations_by_image[image_url]:
            x, y, w, h = item["bbox"]

            # Clip bbox to image boundaries.
            image_info = images[image_id - 1]
            image_width = image_info["width"]
            image_height = image_info["height"]

            x1 = max(0.0, min(x, image_width))
            y1 = max(0.0, min(y, image_height))
            x2 = max(0.0, min(x + w, image_width))
            y2 = max(0.0, min(y + h, image_height))

            clipped_w = x2 - x1
            clipped_h = y2 - y1

            if clipped_w <= 0 or clipped_h <= 0:
                continue

            annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 0,
                "bbox": [x1, y1, clipped_w, clipped_h],
                "area": clipped_w * clipped_h,
                "iscrowd": 0
            })

            annotation_id += 1

    coco = {
        "info": {
            "description": "RF-DETR location tag dataset"
        },
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [
            {
                "id": 0,
                "name": TARGET_CLASS,
                "supercategory": "object"
            }
        ]
    }

    return coco

## 16. Create train COCO annotation

In [ ]:
for split_name in ["train", "valid", "test"]:
    coco_data = build_coco_for_split(split_name, split_urls[split_name])
    ann_path = dataset_root / split_name / "_annotations.coco.json"

    with open(ann_path, "w", encoding="utf-8") as f:
        json.dump(coco_data, f, indent=2)

    print(
        f"{split_name}: "
        f"{len(coco_data['images'])} images, "
        f"{len(coco_data['annotations'])} annotations"
    )

## 17. Save class metadata

In [ ]:
classes_path = dataset_root / "classes.json"

with open(classes_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "num_classes": NUM_CLASSES,
            "classes": [TARGET_CLASS],
            "category_id": 0
        },
        f,
        indent=2
    )

print(f"Saved: {classes_path}")

## 18. Validate dataset structure

In [ ]:
for split in ["train", "valid", "test"]:
    split_dir = dataset_root / split
    images_dir = split_dir / "images"
    ann_path = split_dir / "_annotations.coco.json"

    assert split_dir.exists()
    assert images_dir.exists()
    assert ann_path.exists()

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    assert len(coco["categories"]) == 1
    assert coco["categories"][0]["id"] == 0
    assert coco["categories"][0]["name"] == TARGET_CLASS

    for image_info in coco["images"]:
        image_path = split_dir / image_info["file_name"]
        assert image_path.exists(), f"Missing image: {image_path}"

print("Dataset validation passed.")

## 19. Verify pretrained RF-DETR weights

In [ ]:
weights_path = Path(PRETRAINED_WEIGHTS)

if not weights_path.exists():
    raise FileNotFoundError(f"Pretrained weights not found: {weights_path}")

print(f"Pretrained weights: {weights_path}")
print(f"Size: {weights_path.stat().st_size / (1024**2):.2f} MB")

## 20. Initialize RF-DETR Base

In [ ]:
model = RFDETRBase(
    pretrain_weights=PRETRAINED_WEIGHTS,
    resolution=RESOLUTION,
    device=DEVICE,
    num_classes=NUM_CLASSES
)

print("RF-DETR model initialized.")
print(f"num_classes = {NUM_CLASSES}")
print(f"resolution = {RESOLUTION}")
print(f"device = {DEVICE}")

## 21. Build training configuration

In [ ]:
train_kwargs = {
    "dataset_dir": DATASET_DIR,
    "train_split": "train",
    "val_split": "valid",
    "annotation_file": "_annotations.coco.json",
    "image_folder": "images",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "output_dir": OUTPUT_DIR
}

print(json.dumps(train_kwargs, indent=2))

## 22. Train RF-DETR

In [ ]:
model.train(**train_kwargs)

## 23. Find training checkpoints

In [ ]:
output_root = Path(OUTPUT_DIR)

checkpoint_paths = sorted(
    output_root.rglob("*.pth"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

for checkpoint in tqdm(checkpoint_paths, desc="Searching checkpoints"):
    print(checkpoint)

if not checkpoint_paths:
    raise FileNotFoundError(f"No .pth checkpoints found under {OUTPUT_DIR}")

## 24. Select best checkpoint

In [ ]:
preferred_names = [
    "checkpoint_best_regular.pth",
    "checkpoint_best.pth",
    "best.pth"
]

BEST_CHECKPOINT = None

for name in preferred_names:
    matches = list(output_root.rglob(name))
    if matches:
        BEST_CHECKPOINT = matches[0]
        break

if BEST_CHECKPOINT is None:
    BEST_CHECKPOINT = checkpoint_paths[0]

print(f"Selected checkpoint: {BEST_CHECKPOINT}")

## 25. Load trained model for inference

In [ ]:
trained_model = RFDETRBase(
    pretrain_weights=str(BEST_CHECKPOINT),
    resolution=RESOLUTION,
    device=DEVICE,
    num_classes=NUM_CLASSES
)

print("Trained model loaded.")

## 26. Optional inference optimization

In [ ]:
# Optional. Uncomment if supported by your installed RF-DETR version.
# trained_model.optimize_for_inference()

## 27. Test inference configuration

In [ ]:
test_image_dir = dataset_root / "test" / "images"
inference_output_dir = Path(INFERENCE_OUTPUT_DIR)
inference_output_dir.mkdir(parents=True, exist_ok=True)

test_image_paths = sorted(
    path for path in test_image_dir.iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
)

print(f"Test images: {len(test_image_paths)}")
print(f"Confidence threshold: {CONFIDENCE}")

## 28. Run inference on test images

In [ ]:
inference_results = {}

for image_path in tqdm(test_image_paths, desc="Running test inference"):
    image = Image.open(image_path).convert("RGB")

    detections = trained_model.predict(
        image,
        threshold=CONFIDENCE
    )

    inference_results[image_path.name] = detections

print("Test inference completed.")

## 29. Visualize one test prediction

In [ ]:
if not test_image_paths:
    raise RuntimeError("No test images available.")

preview_path = test_image_paths[0]
image = Image.open(preview_path).convert("RGB")
detections = inference_results[preview_path.name]

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

labels = [
    f"{TARGET_CLASS} {confidence:.2f}"
    for confidence in detections.confidence
]

annotated = box_annotator.annotate(
    scene=np.array(image),
    detections=detections
)

annotated = label_annotator.annotate(
    scene=annotated,
    detections=detections,
    labels=labels
)

annotated_image = Image.fromarray(annotated)
annotated_image.thumbnail((1000, 1000))
annotated_image

## 30. Save annotated test predictions

In [ ]:
saved_count = 0

for image_path in tqdm(test_image_paths, desc="Saving annotated predictions"):
    image = Image.open(image_path).convert("RGB")
    detections = inference_results[image_path.name]

    labels = [
        f"{TARGET_CLASS} {confidence:.2f}"
        for confidence in detections.confidence
    ]

    annotated = box_annotator.annotate(
        scene=np.array(image),
        detections=detections
    )

    annotated = label_annotator.annotate(
        scene=annotated,
        detections=detections,
        labels=labels
    )

    output_path = inference_output_dir / image_path.name
    Image.fromarray(annotated).save(output_path)
    saved_count += 1

print(f"Saved {saved_count} annotated images to {inference_output_dir}")

## 31. Print detection summary

In [ ]:
total_detections = 0

for image_name, detections in inference_results.items():
    count = len(detections)
    total_detections += count

    print(f"{image_name}: {count} detection(s)")

print(f"Total detections: {total_detections}")

## 32. Final paths

In [ ]:
print("Dataset:", Path(DATASET_DIR).resolve())
print("Training output:", Path(OUTPUT_DIR).resolve())
print("Best checkpoint:", Path(BEST_CHECKPOINT).resolve())
print("Inference output:", Path(INFERENCE_OUTPUT_DIR).resolve())
print("Class:", TARGET_CLASS)
print("Classes:", NUM_CLASSES)
print("Resolution:", RESOLUTION)
print("Device:", DEVICE)